In [1]:
import sunpy
import numpy as np
from math import *
import astropy.units as u
from astropy.io import fits
import matplotlib.pyplot as plt
from sunpy.coordinates import frames
import matplotlib.gridspec as gridspec
from scipy.interpolate import interp1d,RegularGridInterpolator
from astropy.coordinates import SkyCoord
from scipy.io import readsav
from matplotlib.patches import Rectangle
from scipy.ndimage import zoom
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_skycoord
import cv2
import glob
import os
from astropy.coordinates import SkyCoord
from sunpy.physics.differential_rotation import differential_rotate
from sunpy.coordinates import propagate_with_solar_surface
import pandas as pd

In [2]:
df=pd.read_excel('../dataset/X级耀斑.xlsx')
df.drop(df.index[8:16], inplace=True)

In [3]:
root_dir=r'C:\Learning\PHD2nd\sunspotscar\data\X'
file_time=os.listdir(root_dir)
df=pd.read_excel('../dataset/X级耀斑.xlsx')
df.drop(df.index[8:16], inplace=True)
x_range=df['X, arcsec'].values
y_range=df['Y, arcsec'].values
passbands=['131','211','304','hmi.Ic_45s']

for i in range(len(file_time)):
    for j in passbands:
        file_name=glob.glob(os.path.join(root_dir,file_time[i],j,'*.fits'))
        os.makedirs(os.path.join(root_dir,file_time[i],j+'sub_map'), exist_ok=True)
        sub_file_path=os.path.join(root_dir,file_time[i],j+'sub_map')
        for k in range(len(file_name)):
            if k==0:
                map0=sunpy.map.Map(file_name[k])
                bl1=SkyCoord((x_range[k]-200)*u.arcsec,(y_range[k]-200)*u.arcsec, frame=map0.coordinate_frame)
                tr1=SkyCoord((x_range[k]+200)*u.arcsec,(y_range[k]+200)*u.arcsec, frame=map0.coordinate_frame)
                sub_map0=map0.submap(bl1,top_right=tr1)
                sub_map0.save(os.path.join(sub_file_path,file_name[k][-38:]))
            else:
                map1=sunpy.map.Map(file_name[k])
                with propagate_with_solar_surface():
                    rot_map1=map1.reproject_to(sub_map0.wcs,preserve_date_obs=True)
                rot_map1 = map1.reproject_to(sub_map0.wcs, preserve_date_obs=True)
                for key in ['telescop', 'instrume', 'detector', 'wavelnth', 'waveunit',
                            'bunit', 'exptime']:
                    if key in map1.meta:
                        rot_map1.meta[key] = map1.meta[key]

                rot_map1 = sunpy.map.Map(rot_map1.data.astype('float32'), rot_map1.meta)
                rot_map1.save(os.path.join(sub_file_path, file_name[k][-38:]), overwrite=True)



In [ ]:
ssmap0=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131sub_map\2012-07-12T150710Z.131.image_lev1.fits')
im0=ssmap0.plot(cmap='sdoaia131')
plt.colorbar(im0)

In [ ]:
ssmap1=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131sub_map\2012-07-12T150746Z.131.image_lev1.fits')
im1=ssmap1.plot(cmap='sdoaia131')
plt.colorbar(im1)

In [ ]:
map1=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131\aia.lev1_euv_12s.2012-07-12T150710Z.131.image_lev1.fits')
map2=sunpy.map.Map(r'C:\Learning\PHD2nd\sunspotscar\data\X\20120712_153700_UTC\131\aia.lev1_euv_12s.2012-07-12T154646Z.131.image_lev1.fits')

In [ ]:
bl1=SkyCoord(Tx=(63.3-100)*u.arcsec,Ty=-436.8*u.arcsec,frame=map1.coordinate_frame)
tr1=SkyCoord(Tx=163.3*u.arcsec,Ty=-266*u.arcsec,frame=map1.coordinate_frame)
smap1=map1.submap(bl1,top_right=tr1)
smap1.plot(cmap='sdoaia131')

In [ ]:
with propagate_with_solar_surface():
    rot_map22=map2.reproject_to(smap1.wcs,preserve_date_obs=True)
rot_map22.plot(cmap='sdoaia131')